## CardiaSense AI — Preprocessing & leakage-safe splits

This notebook covers: **DSP, augmentation, grouped cross-validation splits**.

Run `./setup.sh` first (creates the venv, installs deps, downloads the dataset if needed).
The dataset root is resolved from `CARDIASENSE_DATA_DIR` or `.env`.

> Educational / research prototype — not a medical device.


In [1]:

import os, sys
from pathlib import Path

ROOT = Path("/home/tnzr/CardiaSense-AI")
sys.path.insert(0, str(ROOT / "ml"))
sys.path.insert(0, str(ROOT))

DATA = os.environ.get("CARDIASENSE_DATA_DIR") or     "/media/tnzr/AuxVolume/datasets/HLS-CMDS Heart and Lung Sounds Dataset " \
    "Recorded from a Clinical Manikin using Digital Stethoscope"


### Band-pass processing (heart vs lung bands)

In [2]:

import numpy as np
from cardia.config import Config
from cardia.data.transforms import load_wav, peak_normalize, preprocess

cfg = Config()
row = __import__("cardia.data.hls_cmds", fromlist=["build_task_manifest"]).build_task_manifest(DATA, "lung", "binary").iloc[0]
raw, sr = load_wav(row["file_path"]); raw = peak_normalize(raw)
filt = preprocess(row["file_path"], cfg)   # lung band 60-1500 Hz
from cardia.wandb_utils import viz
viz.overlay_fig(raw.numpy(), filt.numpy(), sr, f"band-pass overlay {row['sample_id']}")
plt = __import__("matplotlib.pyplot", fromlist=["show"]); plt.show()


/tmp/ipykernel_963338/3385638290.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt = __import__("matplotlib.pyplot", fromlist=["show"]); plt.show()


### Augmentation (train-only, seeded)

In [3]:

from cardia.data.transforms import Augment, MelSpec
aug = Augment(cfg, cfg.seed); aug.set_epoch(0)
w = aug.waveform(filt); spec = mel_spec = MelSpec(cfg.sample_rate, cfg.n_mels, cfg.n_fft, cfg.hop_length)
viz.aug_fig(mel_spec(filt).numpy(), aug.spec(mel_spec(w)).numpy(), "before / after augmentation")
plt.show()


/tmp/ipykernel_963338/1385210899.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Leakage-safe grouped splits

In [4]:

from cardia.data.splits import make_splits
import pandas as pd
m = __import__("cardia.data.hls_cmds", fromlist=["build_task_manifest"]).build_task_manifest(DATA, "sound", "binary")
splits = make_splits(m, n_folds=5, seed=0)   # asserts train/val/test group disjointness
rows = [{"fold": s.fold, "train": len(s.train), "val": len(s.val), "test": len(s.test)} for s in splits]
print(pd.DataFrame(rows).to_string(index=False))


 fold  train  val  test
    0    321  107   107
    1    321  107   107
    2    321  107   107
    3    321  107   107
    4    321  107   107
